<a href="https://colab.research.google.com/github/martinezfredy33/infografia/blob/main/AN%C3%81LISIS_DESCRIPTIVO_DATOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Práctico ETL: Limpieza e Higienización de Datos

Asignatura: Business Analytics (BA)

Dataset Base: S3_dataset_con_errores_Mineria_UDEC.csv

Objetivo: Aplicar la fase de Extracción, Transformación y Carga (ETL) sobre una base de datos con inconsistencias, nulos y errores de formato utilizando Pandas en Google Colab.

# Fase 1: Extracción (Extract)

In [17]:
import pandas as pd
import numpy as np
# 1. Cargar el archivo CSV
ruta_archivo = '/content/S3_dataset_con_errores_Mineria_UDEC.csv'
try:
  df = pd.read_csv(ruta_archivo, sep=';')
  print("Archivo cargado exitosamente con delimitador ';'")
except Exception: df = pd.read_csv(ruta_archivo, sep=',')
print("Archivo cargado exitosamente con delimitador ','")
# Limpieza preventiva de espacios en los nombres de las columnas
df.columns = df.columns.str.strip()
print(f"Dimensiones iniciales: {df.shape[0]} filas y {df.shape[1]} columnas.")

Archivo cargado exitosamente con delimitador ';'
Archivo cargado exitosamente con delimitador ','
Dimensiones iniciales: 522 filas y 12 columnas.


#Diagnóstico Inicial

Diagnóstico e Inspección de Inconsistencias Se verifica el estado inicial del dataset para cuantificar nulos, filas duplicadas y tipos de datos asignados por Pandas.

In [18]:
# Muestra de los primeros registros
display(df.head())
# Diagnóstico de valores nulos
print("\\n--- Conteo de Valores Nulos por Columna ---")
print(df.isnull().sum())
# Diagnóstico de filas duplicadas
print(f"\\nNúmero de filas exactamente duplicadas: {df.duplicated().sum()}")

,Codigo Cliente,Nombre Cliente,Apellidos Cliente,Direccion,Telefono,Ciudad,Pais,SKU Producto,Cantidad,Descripcion Producto,Fecha de Compra,Fecha de Entrega
0,C001,Peggy,Wilson,252 Jennifer Motorway Apt. 026 South Mitchellm...,1997841929,West Patricktown,Tunisia,SKU-1008,100,Webcam,2024-06-03,2024-08-18
1,C001,Peggy,Wilson,252 Jennifer Motorway Apt. 026 South Mitchellm...,1997841929,West Patricktown,Tunisia,SKU-1008,5,Webcam,2024-06-28,2024-08-04
2,C002,Andrea,Miles,NaN,(140)892-5838,Carterfort,Saint Kitts and Nevis,SKU-1003,3,Headphones,2022-09-17,2024-09-05
3,C002,Andrea,Miles,NaN,(140)892-5838,Carterfort,Saint Kitts and Nevis,SKU-1007,100,Printer,2024-05-28,2024-06-10
4,C003,Kristen,Yates,"0493 Timothy Islands Smithmouth, MT 09836",(638)827-4539x7556,West Laurabury,Dominican Republic,SKU-1008,10,Webcam,2024-01-25,2024-03-04


\n--- Conteo de Valores Nulos por Columna ---
Codigo Cliente           0
Nombre Cliente           0
Apellidos Cliente        0
Direccion               61
Telefono                 0
Ciudad                   0
Pais                     0
SKU Producto             0
Cantidad                 3
Descripcion Producto     0
Fecha de Compra          0
Fecha de Entrega        10
dtype: int64
\nNúmero de filas exactamente duplicadas: 0


#Fase 2: Transformación (Transform)

Se aplican las reglas de negocio e higienización especificas para este dataset:
1. Estandarización de Texto: Remoción de espacios invisibles (strip()) y formato de mayúscula inicial (title()).
2. Limpieza de Cantidades: Conversión a tipo numérico, corrección de espacios vacíos, imputación de nulos y conversión de valores negativos a positivos mediante valor absoluto (abs()).
3. Conversión de Fechas: Transformación de Fecha de Compra y Fecha de Entrega a formato de fecha (datetime).
4. Campos Calculados: Nombre_Completo: Concatenación de Nombre y Apellidos. Dias_Entrega: Días transcurridos entre la compra y la entrega ((Fecha de Entrega - Fecha de Compra).dt.days).
5. Tratamiento de Direcciones Faltantes: Imputación de nulos en Direccion con el valor 'No Especificada'.

##Aplicación de Transformaciones

In [19]:
import pandas as pd
import numpy as np

# 1. Estandarización selectiva de columnas de texto (excluyendo SKUs o códigos si aplica)
columnas_categoricas = ['Nombre Cliente', 'Apellidos Cliente', 'Ciudad', 'Pais', 'Direccion']
for col in columnas_categoricas:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# Reemplazo estricto de nulos textuales o espacios vacíos
nulos_textuales = ['nan', 'NaN', 'None', '', ' ']
df.replace(nulos_textuales, np.nan, inplace=True)

# 2. Eliminación de duplicados
df = df.drop_duplicates()

# 3. Corrección y limpieza de la columna 'Cantidad'
df['Cantidad'] = pd.to_numeric(df['Cantidad'], errors='coerce')
# Evaluar si rellenar con 0 sesga las métricas de inventario o ventas;
# a veces es preferible mantener NaN o imputar con la mediana según el contexto analítico.
df['Cantidad'] = df['Cantidad'].fillna(0).abs()

# 4. Conversión de Fechas
df['Fecha de Compra'] = pd.to_datetime(df['Fecha de Compra'], errors='coerce')
df['Fecha de Entrega'] = pd.to_datetime(df['Fecha de Entrega'], errors='coerce')

# 5. Tratamiento de nulos en Dirección
if 'Direccion' in df.columns:
    df['Direccion'] = df['Direccion'].fillna('No Especificada')

# 6. Creación de Columnas Calculadas asegurando la lógica temporal
if 'Nombre Cliente' in df.columns and 'Apellidos Cliente' in df.columns:
    df['Nombre_Completo'] = df['Nombre Cliente'] + ' ' + df['Apellidos Cliente']

# Cálculo de días de entrega controlando anomalías temporales (si entrega < compra, asignar NaN)
dias_delta = (df['Fecha de Entrega'] - df['Fecha de Compra']).dt.days
df['Dias_Entrega'] = np.where(dias_delta >= 0, dias_delta, np.nan)
df['Dias_Entrega'] = df['Dias_Entrega'].fillna(0)

print("Transformaciones validadas y completadas.")
print("\n--- Resumen del Dataset Depurado ---")
display(df[['Codigo Cliente', 'Nombre_Completo', 'Ciudad', 'Pais', 'SKU Producto', 'Cantidad', 'Fecha de Compra', 'Fecha de Entrega', 'Dias_Entrega']].head())

Transformaciones validadas y completadas.

--- Resumen del Dataset Depurado ---


,Codigo Cliente,Nombre_Completo,Ciudad,Pais,SKU Producto,Cantidad,Fecha de Compra,Fecha de Entrega,Dias_Entrega
0,C001,Peggy Wilson,West Patricktown,Tunisia,SKU-1008,100.0,2024-06-03,2024-08-18,76.0
1,C001,Peggy Wilson,West Patricktown,Tunisia,SKU-1008,5.0,2024-06-28,2024-08-04,37.0
2,C002,Andrea Miles,Carterfort,Saint Kitts And Nevis,SKU-1003,3.0,2022-09-17,2024-09-05,719.0
3,C002,Andrea Miles,Carterfort,Saint Kitts And Nevis,SKU-1007,100.0,2024-05-28,2024-06-10,13.0
4,C003,Kristen Yates,West Laurabury,Dominican Republic,SKU-1008,10.0,2024-01-25,2024-03-04,39.0


#Fase 3: Carga (Load)

## Se exporta el DataFrame completamente depurado hacia un nuevo archivo plano CSV estandarizado (S3_dataset_con_errores_limpio_ETL.csv).

"index=False": Evita la creación de la columna implícita de índices de Pandas.

encoding="utf-8-sig": Mantiene la compatibilidad de tildes y eñes para su correcta lectura en Microsoft Excel y Power BI.

##Exportación Final

In [20]:
# Exportar el resultado final a CSV
ruta_salida = '/content/S3_dataset_con_errores_limpio_ETL.csv'
df.to_csv(ruta_salida, index=False, sep=';', encoding='utf-8-sig')
print(f"Proceso ETL finalizado con éxito.")
print(f"El dataset limpio se guardó en: {ruta_salida}")

Proceso ETL finalizado con éxito.
El dataset limpio se guardó en: /content/S3_dataset_con_errores_limpio_ETL.csv
